In [1]:
# imports
import pandas as pd
from tensorflow.keras import layers, Model
# utilities
import numpy as np

In [2]:
# Separate features into different categories
X_numerical = ['INCL', 'RAAN', 'ECC', 'ARG_PER', 'MEAN_MOTION', 'SMA_KM','APOGEE_KM', 'PERIGEE_KM', 'MEAN_MOTION_1ST_DER'] 
X_numerical_revised = X_numerical + ['MEAN_MOTION_1ST_DER', 'B_STAR']
response = 'TYPE'

# Fetch final dataframe
df_final = pd.read_csv('../data/large/final_df_v1.csv') # <-- Run Part 1 of DataConsolidation.ipynb to produce file
df_final = df_final.sort_values(by=['NUMBER', 'EPOCH'])
df_final.reset_index(inplace=True, drop=True)

In [3]:
X_features_to_use = X_numerical
N_FEAT = len(X_features_to_use)
STATE_COUNT_THRESHOLD = 20 # <-- should tune this

# 1. count number of states per object
rso2count = df_final['NUMBER'].value_counts()

# 2. choose threshold and find RSOs that meet the requirement
rsos_geq_threshold = rso2count[rso2count.values >= STATE_COUNT_THRESHOLD].index
NUM_OBJECTS = len(rsos_geq_threshold)
print(NUM_OBJECTS) 
print(len(rsos_geq_threshold)/len(rso2count.values ))

# 3. TODO scaling train/test split, reshape
# End with an np array of shape (NUM_OBJECTS, STATE_COUNT_THRESHOLD, N_FEAT)

23161
0.7595015576323988


In [5]:
X_train = np.random.rand(10000, STATE_COUNT_THRESHOLD, N_FEAT).astype(np.float32)  # or float64 if needed
y_train = np.random.randint(0, 3, size=(10000,))

def single_state_evaluation_model(): 
    '''given features of ONE state, returns softmax probs'''

    inputs = layers.Input(shape=(N_FEAT,))
    x = layers.Dense(64, activation='relu')(inputs)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(3, activation='softmax')(x)
    return Model(inputs, outputs, name="StateModel")


def create_object_model(state_model):
    '''per object model'''
    # STATE_COUNT_THRESHOLD states per object, each of shape (N_FEAT,)
    inputs = layers.Input(shape=(STATE_COUNT_THRESHOLD, N_FEAT))
    # apply the state model to each of the STATE_COUNT_THRESHOLD states
    state_probs = layers.TimeDistributed(state_model)(inputs)  # Output shape: (20, 3)

    # classify the object based on these STATE_COUNT_THRESHOLD softmaxed vectors
    x = layers.Flatten()(state_probs)  # Shape: (3*STATE_COUNT_THRESHOLD,)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(3, activation='softmax')(x)

    return Model(inputs, outputs, name="ObjectClassifier")


state_model = single_state_evaluation_model()
object_model = create_object_model(state_model)

object_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# `X` is (num_objects, STATE_COUNT_THRESHOLD, N_FEAT) and `y` is (num_objects,)
object_model.fit(X_train, y_train, batch_size=32, epochs=3)




Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.3424 - loss: 1.1049
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.3370 - loss: 1.1018
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.3410 - loss: 1.1034
